In [10]:
import pandas as pd
from transformers import pipeline
import os  

print("="*60)
print("KR-FinBERT 로드 중 (뉴스/사업보고서 전용)")
finbert = pipeline("text-classification", model="snunlp/KR-FinBert-SC", device=-1)
print("="*60)

# 긍정/부정 결과를 0.0 ~ 1.0 사이의 수치로 나타냄
def get_sentiment_score(text):
    try:
        # KR-FinBert 모델의 최대 토큰이 512이므로 안전장치 적용
        result = finbert(text, truncation=True, max_length=512)[0]
        label = result['label'].lower()
        score = result['score'] 
        
        # 긍정일 경우(예: 긍정 0.9 -> 0.9)
        if 'pos' in label or '1' in label:
            result_score = score
        # 부정일 경우: (예: 부정 0.9 -> 0.1)
        elif 'neg' in label or '0' in label:
            result_score = 1.0 - score
        # 중립일 경우: 0.5 
        else:
            return 0.5
        
        return round(result_score, 1)
    
    except Exception as e: # 예외처리도 중립
        return 0.5 

input_folder = "../data/Report"
output_folder = "../data/Report_Positive"

# 가져온 csv 데이터 파일 보고서 긍정값
csv_files = [
    "Hanmi_Semiconductor_2020_2024.csv",
    "Samsung_Electronics_Comprehensive_2020_2024.csv",
    "SK_Hynix_Comprehensive_2020_2024.csv"
]


for file_name in csv_files:
    print(f"\n [{file_name}] 파일 탐색")

    input_filepath = os.path.join(input_folder, file_name)

    try:
        df = pd.read_csv(input_filepath)

        print("감성 분석 중")
        df['보고서_긍정값'] = df['Unstructured_Text_Content'].apply(get_sentiment_score)
        #data/report_positive 폴더 안에 csv 생성
        output_filename = f"{file_name}"
        output_filepath = os.path.join(output_folder, output_filename)

        # 생성한 경로에 파일 저장
        df.to_csv(output_filepath, index=False, encoding='utf-8-sig')
        print(f" '{output_filepath}' 경로에 안전하게 저장되었습니다.")
        
    except Exception as e:
        print(f" 에러 발생: {e}")

KR-FinBERT 로드 중 (뉴스/사업보고서 전용)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 56289.98it/s]



 [Hanmi_Semiconductor_2020_2024.csv] 파일 탐색
감성 분석 중
 '../data/Report_Positive\Hanmi_Semiconductor_2020_2024.csv' 경로에 안전하게 저장되었습니다.

 [Samsung_Electronics_Comprehensive_2020_2024.csv] 파일 탐색
감성 분석 중
 '../data/Report_Positive\Samsung_Electronics_Comprehensive_2020_2024.csv' 경로에 안전하게 저장되었습니다.

 [SK_Hynix_Comprehensive_2020_2024.csv] 파일 탐색
감성 분석 중
 '../data/Report_Positive\SK_Hynix_Comprehensive_2020_2024.csv' 경로에 안전하게 저장되었습니다.
